# CMS Blue Button Initial Setup

One-time OAuth authorization flow to obtain and store the refresh token.

**CRITICAL**: CMS sandbox requires logging in **as a synthetic test beneficiary** during OAuth, not just approving the app. Use credentials:
- Username: `BBUser00001`
- Password: `PW00001!`
- Only BBUser00001 verified, others may not work reliably.

Run this notebook once to complete initial setup, then use `CMS_Blue_Button_API_sample_call` for all subsequent API calls.

In [0]:
import requests
import json
import urllib.parse
import base64
import hashlib
import secrets
from databricks.sdk import WorkspaceClient

# Retrieve credentials
client_id = "IyDGHLIUhNe4xblVyCXCnLLzxsB2reD34dDVhrBn"
client_secret = dbutils.secrets.get(scope="cms_blue_button_api", key="client_secret")

BASE_URL = "https://sandbox.bluebutton.cms.gov"
REDIRECT_URI = "http://localhost/callback"

print("Credentials retrieved.")

In [0]:
# Generate PKCE code_verifier and code_challenge
code_verifier = secrets.token_urlsafe(64)
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).decode().rstrip("=")

# Generate state for CSRF protection
state = secrets.token_urlsafe(32)

# Explicitly request correct scopes per CMS Blue Button API documentation
scope = "openid profile patient/Patient.read patient/Coverage.read patient/ExplanationOfBenefit.read"

auth_url = (
    f"{BASE_URL}/v2/o/authorize/"
    f"?response_type=code"
    f"&client_id={client_id}"
    f"&redirect_uri={urllib.parse.quote(REDIRECT_URI, safe='')}"
    f"&scope={urllib.parse.quote(scope, safe='')}"
    f"&code_challenge={code_challenge}"
    f"&code_challenge_method=S256"
    f"&state={state}"
)

print("STEP 1: Log out of the CMS sandbox first!")
print("  Visit https://sandbox.bluebutton.cms.gov/ and log out.")
print("  Being logged in causes the OAuth flow to skip beneficiary login.")
print()
print("STEP 2: Visit this URL in your browser to authorize:")
print()
print(auth_url)
print()
print("STEP 3: When prompted, LOG IN as a synthetic test beneficiary:")
print("  Username: BBUser00001")
print("  Password: PW00001!")
print("Then click 'Connect' / approve the 'Immunication Tracker' app.")
print()
print("STEP 4: After authorizing, your browser redirects to:")
print(f"  {REDIRECT_URI}?code=XXXX&state={state}")
print("Copy the 'code' parameter value and paste it as AUTH_CODE in the next cell.")
print("Verify that the 'state' in the redirect URL matches:")
print(f"  {state}")
print("\nIf you see 'Invalid scopes.' after login, confirm your app registration (in CMS sandbox dashboard) has demographic data enabled and matches these scopes exactly:")
print(scope)
print("Scopes must end with .read suffix, not .rs")

In [0]:
# Paste the authorization code from the redirect URL below
AUTH_CODE = "8v06w5VUrW8VrA7ujnhRoBj9plIuL6"

token_response = requests.post(
    f"{BASE_URL}/v2/o/token/",
    data={
        "grant_type": "authorization_code",
        "code": AUTH_CODE,
        "client_id": client_id,
        "client_secret": client_secret,
        "redirect_uri": REDIRECT_URI,
        "code_verifier": code_verifier,
    },
)

token_data = token_response.json()

if "access_token" in token_data:
    access_token = token_data["access_token"]
    print("✓ Access token retrieved successfully.")
    print(f"Token type: {token_data.get('token_type')}")
    print(f"Expires in: {token_data.get('expires_in')} seconds")
    print(f"Valid until: {token_data.get('access_grant_expiration')}")
else:
    print("Token exchange failed:")
    print(json.dumps(token_data, indent=2))

In [0]:
# Store the refresh token and make the initial access token available
w = WorkspaceClient()

w.secrets.put_secret(
    scope="cms_blue_button_api",
    key="refresh_token",
    string_value=token_data["refresh_token"]
)

print("✓ Refresh token stored in secret scope 'cms_blue_button_api'")
print(f"Valid until: {token_data['access_grant_expiration']}")
print()
print("IMPORTANT: The access_token variable is now available in this notebook's namespace.")
print("Run the next cell to make API calls using this initial access token.")
print("After 1 hour, use the CMS_Blue_Button_API_sample_call notebook to refresh.")

In [0]:
# Make sample FHIR API calls using the access token from the initial OAuth exchange
# This validates that the setup worked correctly

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/fhir+json",
}

# Patient resource
print("=" * 60)
print("Patient Resource")
print("=" * 60)
patient_resp = requests.get(f"{BASE_URL}/v2/fhir/Patient", headers=headers)
patient_data = patient_resp.json()
print(json.dumps(patient_data, indent=2))

# Coverage resources  
print("\n" + "=" * 60)
print("Coverage Resources")
print("=" * 60)
coverage_resp = requests.get(f"{BASE_URL}/v2/fhir/Coverage", headers=headers)
coverage_data = coverage_resp.json()
print(json.dumps(coverage_data, indent=2))

# ExplanationOfBenefit resources
print("\n" + "=" * 60)
print("ExplanationOfBenefit Resources")
print("=" * 60)
eob_resp = requests.get(f"{BASE_URL}/v2/fhir/ExplanationOfBenefit", headers=headers)
eob_data = eob_resp.json()
print(json.dumps(eob_data, indent=2))